In [0]:
df = spark.read.table("default.bronze_breweries")
display(df)

In [0]:
from pyspark.sql import functions as F

(
    df
    .select(
            'address_1', 
            'address_2', 
            'address_3', 
            'brewery_type', 
            'city', 
            'country', 
            'id', 
            'latitude', 
            'longitude', 
            'name', 
            'phone', 
            'postal_code', 
            'state', 
            'state_province', 
            'street', 
            'website_url'
    )
    .where(
        (F.col("country") == "United States") &
        (F.col("address_1").isNotNull())         
    )
    .withColumn(
        "address_2",
        F.when(
            F.col("address_2").isNull(),
            F.lit("Doesn't exist")
        ).otherwise( 
            F.col("address_2")
        )
    )
    .withColumn(
        "address_3",
        F.when(
            F.col("address_3").isNull(),
            F.lit("Doesn't exist")
        ).otherwise( 
            F.col("address_3")
        )
    )
    .withColumn(
        "name",
        F.regexp_replace(F.col("name"), "Â", "")
    )
    .withColumn(
        "phone",
        F.when(
            F.col('phone').isNull(), 
            F.lit("Unknown")
        ).otherwise(
            F.regexp_replace((F.col('phone')), r'^\+\d{1,3}\s', '')
        )
    )
    .withColumn(
        "phone",
        F.regexp_replace(F.col("phone"), r"[^\d]", "") 
    )
    .withColumn(
        "postal_code",
        F.when(
             F.col("postal_code").isNull(), 
             F.lit("Unknown")
        )
        .when(
            F.col("country") == "United States",
            F.regexp_replace(F.col("postal_code"), r"-.*", "")
        )
        .otherwise(
            F.col("postal_code")
        )
    )
    .withColumn(
        "ingestion_ts", 
        F.current_timestamp() #da utilizzare come sequence_by
    ) 
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "True")
    .saveAsTable("silver_breweries")

)

In [0]:
row_selection = spark.sql("SELECT * FROM silver_breweries")
display(row_selection)